# 真实 IPYNB 的结构化切块与多模态 Embedding 实验

填写课程 `.ipynb` 路径后，本 Notebook 通过项目工具执行：

`NotebookParser → NotebookChunker → DocumentIndexer → Qwen text/VL embedding`

解析过程不会执行课程 Notebook 中的代码。HTML/DataFrame 表格使用文本 embedding；图片、图表和图片表格使用 `qwen3-vl-embedding` 图文融合向量。

In [ ]:
from collections import Counter
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from app.config import get_settings
from app.rag.chunker import NotebookChunker
from app.rag.indexer import DocumentIndexer
from app.rag.parsers.notebook import NotebookParser

# 填写绝对路径或项目根目录下的相对路径。
NOTEBOOK_PATH = ''

settings = get_settings()
parser = NotebookParser.from_settings(settings)
chunker = NotebookChunker.from_settings(settings)
indexer = DocumentIndexer.from_settings(settings)

In [ ]:
if not NOTEBOOK_PATH.strip():
    raise ValueError('请先在 NOTEBOOK_PATH 中填写一个 .ipynb 文件路径')

source_path = Path(NOTEBOOK_PATH).expanduser()
if not source_path.is_absolute():
    source_path = PROJECT_ROOT / source_path

document = parser.parse(source_path)
print(f'文档标题: {document.title}')
print(f'文档 ID: {document.document_id}')
print(f'来源文件: {document.source_path}')
print(f'内容块: {len(document.blocks)}')
print(f'类型统计: {Counter(block.kind.value for block in document.blocks)}')
print(f'图片资源: {sum(len(block.assets) for block in document.blocks)}')

In [ ]:
structure = chunker.chunk(document)

print(f'Cell Group: {len(structure.groups)}')
print(f'标题段: {len(structure.sections)}')
print(f'Chunk: {len(structure.chunks)}')
print(f'Chunk 类型: {Counter(chunk.kind.value for chunk in structure.chunks)}')
print(f'Token 范围: {min(c.token_count for c in structure.chunks)} - {max(c.token_count for c in structure.chunks)}')
print(f'超过硬上限: {sum(c.token_count > settings.notebook_chunk_max_tokens for c in structure.chunks)}')

for index, chunk in enumerate(structure.chunks[:20]):
    preview = chunk.embedding_text.replace('\n', ' ')[:150]
    print(
        f'chunk[{index}] kind={chunk.kind.value}, tokens={chunk.token_count}, '
        f'section={" > ".join(chunk.section_path)}, cells={chunk.cell_ids} | {preview}'
    )

In [ ]:
prepared = await indexer.prepare_document(document)

text_types = Counter(
    item.representation.representation_type.value
    for item in prepared.text_representations
)
print(f'文本表示: {len(prepared.text_representations)} {text_types}')
print(f'文本模型: {prepared.text_embedding_model}')
print(f'文本向量维度: {prepared.text_embedding_dimension}')
print(f'文本 Token: {prepared.text_total_tokens}')
print(f'视觉融合表示: {len(prepared.visual_representations)}')
print(f'视觉模型: {prepared.visual_embedding_model}')
print(f'视觉向量维度: {prepared.visual_embedding_dimension}')
print(f'视觉 Token: {prepared.visual_total_tokens}')
print(f'跳过的视觉资源: {prepared.skipped_visual_asset_ids}')

for item in prepared.visual_representations[:10]:
    rep = item.representation
    print(
        f'{rep.source_id} | {rep.media_type} | chunk={rep.chunk_id} | '
        f'dimension={len(item.dense)} | {rep.image_path}'
    )

## 检查重点

- 一个代码 cell 与它的输出是否处于同一个 `execution` group。
- Chunk 是否只属于一个标题段，且不超过硬上限。
- `document_outline` 是否只出现一次，`text_content` 是否与 chunk 一一对应。
- HTML/DataFrame 表格是否进入文本表示。
- PNG/JPEG/WEBP/BMP/TIFF 是否生成 `visual_fusion`；SVG/GIF 当前会记录为跳过资源。